# Team Generalization

In this notebook, we apply the trained models to the test dataset to generate final predictions.

The steps include:
- applying the same preprocessing used during training  
- assigning each company to a cluster  
- using the corresponding cluster model to predict bankruptcy  


## 1. Setup

Importing libraries and adding a couple of compatibility shims because some teammates saved their models with sklearn 1.6.1 and others used 1.8.0.

In [22]:
import sys
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

# sklearn version compatibility
# Ozzie's C4 model was saved with sklearn 1.6.1 which had _RemainderColsList
import sklearn.compose._column_transformer as ct
if not hasattr(ct, '_RemainderColsList'):
    class _RemainderColsList(list):
        pass
    ct._RemainderColsList = _RemainderColsList

# Older sklearn LR had a multi_class attribute — set a default so newer sklearn
# doesn't crash when checking it
from sklearn.linear_model import LogisticRegression
LogisticRegression.multi_class = 'auto'

# Add each cluster folder to path so we can import their helper modules
for c in range(7):
    sys.path.insert(0, f'../phase-2/cluster-{c}')

## 2. Load test data

In [23]:
import os

# Try a few paths to find test_data.csv (handles running from different locations)
for p in ['Data/test_data.csv', '../Data/test_data.csv']:
    if os.path.exists(p):
        test_data_path = p
        break

test_df = pd.read_csv(test_data_path)
# Test data has leading spaces in column names, but trained models expect no spaces
test_df.columns = test_df.columns.str.strip()
print(f'Loaded test data from: {test_data_path}')
print(f'Test data: {test_df.shape}')
test_df.head()

Loaded test data from: ../Data/test_data.csv
Test data: (1012, 96)


,Index,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,...,Net Income to Total Assets,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability
0,0,0.414323,0.481029,0.468280,0.609514,0.609514,0.998889,0.797159,0.809132,0.303290,...,0.761704,0.001404,0.623973,0.609512,0.838286,0.275450,0.026749,0.564950,1,0.136203
1,1,0.497441,0.560892,0.546603,0.610660,0.610660,0.999108,0.797545,0.809431,0.303506,...,0.815244,0.004466,0.623724,0.610658,0.842427,0.285886,0.026965,0.565870,1,0.018871
2,2,0.501584,0.548899,0.556721,0.606134,0.606134,0.999034,0.797427,0.809370,0.303453,...,0.806318,0.000684,0.625387,0.606132,0.840598,0.275816,0.026793,0.565165,1,0.095511
3,3,0.574465,0.637375,0.619680,0.600376,0.600376,0.999030,0.797528,0.809426,0.303640,...,0.852655,0.001718,0.624151,0.600375,0.844727,0.279977,0.026795,0.565178,1,0.028513
4,4,0.393360,0.456444,0.440334,0.600009,0.600009,0.998800,0.797025,0.809000,0.303240,...,0.741604,0.002545,0.623612,0.600009,0.835578,0.279901,0.026623,0.564204,1,0.028779


## 3. Route test companies to clusters

Khush's classifier predicts which cluster each test company belongs to. 

The same preprocessing steps used during training are applied here.

This includes:
- dropping unnecessary columns like Index  
- ensuring the feature set matches what the models expect  


In [24]:
router = joblib.load('../Data/cluster_id_classifier.joblib')

# Drop columns the router doesn't use
X_route = test_df.drop(columns=['Index', 'Net Income Flag'], errors='ignore')

predicted_clusters = router.predict(X_route)

print('Routing distribution:')
for c in sorted(set(predicted_clusters)):
    n = (predicted_clusters == c).sum()
    print(f'  Cluster {c}: {n} test companies')

Routing distribution:
  Cluster 0: 110 test companies
  Cluster 1: 320 test companies
  Cluster 2: 162 test companies
  Cluster 3: 177 test companies
  Cluster 4: 3 test companies
  Cluster 5: 149 test companies
  Cluster 6: 1 test companies
  Cluster 7: 90 test companies


After routing, the test data is split based on cluster IDs so that each cluster model can be applied separately.

## 4. Load helper modules

Each member saved their model with custom Python classes. We need to import those classes before joblib can load the models.

Jeff's C2 and C3 both have a class named `Predictor` so we load each one fresh into the namespace before loading its joblib — that way they don't overwrite each other.

In [25]:
import importlib.util
import __main__

def load_helper(path):
    # Load a .py file fresh and put its classes into __main__
    # so joblib can find them when unpickling
    spec = importlib.util.spec_from_file_location('helper', path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    for name in dir(mod):
        if not name.startswith('_'):
            setattr(__main__, name, getattr(mod, name))

# Load helpers we need by name (used directly later)
load_helper('../phase-2/cluster-0/cluster_0_classes.py')
load_helper('../phase-2/cluster-1/aania_cluster_classes.py')
load_helper('../phase-2/cluster-4/preprocessing.py')
load_helper('../phase-2/cluster-6/cluster_6_classes.py')

print('helpers loaded')

helpers loaded


## 5. Load all cluster models

In [26]:
# C0 (Khush) - wrapped in ClusterPredictor class
from cluster_0_classes import ClusterPredictor as ClusterPredictor0
c0_model = ClusterPredictor0('../phase-2/cluster-0/c0_stacking_model.joblib')

# C1 (Aania) - dict bundle
c1_model = joblib.load('../phase-2/cluster-1/aania_cluster1_model.joblib')

# C2 (Jeff) - load helper first, then joblib
load_helper('../phase-2/cluster-2/cluster_2_classes.py')
c2_model = joblib.load('../phase-2/cluster-2/cluster_2_predictor.joblib')

# C3 (Jeff) - same class names as C2 but different file. The c2_model object
# already exists with its own class refs, so loading C3 doesn't break it.
load_helper('../phase-2/cluster-3/cluster_3_classes.py')
c3_model = joblib.load('../phase-2/cluster-3/cluster_3_predictor.joblib')

# C4 (Ozzie) - sklearn pipeline
c4_model = joblib.load('../phase-2/cluster-4/cluster_4.joblib')

# C5 (Aania) - dict bundle
c5_model = joblib.load('../phase-2/cluster-5/aania_cluster5_model.joblib')

# C6 (Khush) - wrapped in ClusterPredictor class
from cluster_6_classes import ClusterPredictor as ClusterPredictor6
c6_model = ClusterPredictor6('../phase-2/cluster-6/c6_stacking_model.joblib')

# Aania's helper for predicting from dict bundles
from aania_cluster_classes import predict_with_bundle

print('all 7 cluster models loaded')

[C0] Model loaded | features=10 | threshold=0.49
[C6] Model loaded | features=5 | threshold=0.5
all 7 cluster models loaded


## 6. Predict bankruptcy for each cluster

For each cluster, grab the test companies routed to it and run that member's model.

In [27]:
predictions = np.zeros(len(test_df), dtype=int)

print(f'{"Cluster":>8} {"Routed":>8} {"Flagged":>8} {"Rate":>8}')
print('-' * 36)

for c in sorted(set(predicted_clusters)):
    mask = predicted_clusters == c
    n_routed = mask.sum()
    
    # Get the test companies routed to this cluster
    cluster_test = test_df.loc[mask].drop(columns=['Index'])
    
    # Run the right model based on cluster ID
    if c == 0:
        preds = c0_model.predict(cluster_test)
    elif c == 1:
        preds = predict_with_bundle(c1_model, cluster_test)
    elif c == 2:
        preds = c2_model.predict(cluster_test)
    elif c == 3:
        preds = c3_model.predict(cluster_test)
    elif c == 4:
        preds = c4_model.predict(cluster_test)
    elif c == 5:
        preds = predict_with_bundle(c5_model, cluster_test)
    elif c == 6:
        preds = c6_model.predict(cluster_test)
    elif c == 7:
        preds = np.zeros(n_routed, dtype=int)  # constant predictor
    
    preds = np.array(preds, dtype=int)
    predictions[mask] = preds
    
    n_flagged = preds.sum()
    rate = n_flagged / n_routed * 100
    print(f'{c:>8} {n_routed:>8} {n_flagged:>8} {rate:>7.1f}%')

 Cluster   Routed  Flagged     Rate
------------------------------------
       0      110       16    14.5%
       1      320       35    10.9%
       2      162       65    40.1%
       3      177       15     8.5%
       4        3        0     0.0%
       5      149       34    22.8%
       6        1        1   100.0%
       7       90        0     0.0%


## 7. Sparsity check

At most 20% of test companies (202) can be flagged as bankrupt

Combining predictions for all test companies. 


In [28]:
total_flagged = predictions.sum()
total = len(predictions)
limit = int(0.20 * total)

print(f'Total flagged:    {total_flagged} / {total} ({total_flagged/total*100:.2f}%)')
print(f'Sparsity limit:   {limit}')
print(f'Status:           {"PASS" if total_flagged < limit else "OVER BUDGET"}')

if total_flagged >= limit:
    print()
    print('Need to tighten thresholds before submitting.')
    print('Per cluster (largest first):')
    cluster_flags = []
    for c in sorted(set(predicted_clusters)):
        m = predicted_clusters == c
        cluster_flags.append((c, int(predictions[m].sum()), int(m.sum())))
    for c, flagged, n in sorted(cluster_flags, key=lambda x: -x[1]):
        if flagged > 0:
            print(f'  C{c}: {flagged} / {n} ({flagged/n*100:.1f}%)')

Total flagged:    166 / 1012 (16.40%)
Sparsity limit:   202
Status:           PASS


## 8. Save submission CSV

Section 4: `Index, Bankrupt?` columns, one row per test company.



In [29]:
submission = pd.DataFrame({
    'Index': test_df['Index'].values,
    'Bankrupt?': predictions,
})

print(f'Submission: {submission.shape}')
print()
print('First 10 rows:')
print(submission.head(10).to_string(index=False))
print()
print('Value counts:')
print(submission['Bankrupt?'].value_counts())

Submission: (1012, 2)

First 10 rows:
 Index  Bankrupt?
     0          0
     1          1
     2          0
     3          1
     4          0
     5          0
     6          0
     7          1
     8          1
     9          0

Value counts:
Bankrupt?
0    846
1    166
Name: count, dtype: int64


Before saving the results, checked if:
- all test samples have predictions  
- there are no missing values  
- the format matches the required submission structure 

In [30]:
submission.to_csv('4_Generalization.csv', index=False)
print('saved: 4_Generalization.csv')

saved: 4_Generalization.csv
